# Analyse: Szenenklassifikation


In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision pillow numpy pandas tqdm


In [6]:
import os
from pathlib import Path
from collections import Counter
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
from torchvision import models, transforms

print(torch.__version__)
print('CUDA available:', torch.cuda.is_available())


2.7.1+cpu
CUDA available: False


In [7]:
DATA_DIR = Path('../../data')

COMBINED_CSV = DATA_DIR / '03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '06_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_scene_classification.csv'
FRAME_ROOT = DATA_DIR / '02_media/stratified_sample/frames'
SOURCE_FILTER = None
DEFAULT_MAX_FRAMES_PER_VIDEO = 60

MODEL_DIR = Path('../../models/models_cache')
MODEL_WEIGHTS = MODEL_DIR / 'resnet18_places365.pth.tar'
CATEGORIES_TXT = MODEL_DIR / 'categories_places365.txt'

MODEL_URL = 'http://places2.csail.mit.edu/models_places365/resnet18_places365.pth.tar'
CATEGORIES_URL = 'https://raw.githubusercontent.com/csailvision/places365/master/categories_places365.txt'

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Running on device: {device}')

df = pd.read_csv(COMBINED_CSV)
if 'influencer_type' not in df.columns and 'source' in df.columns:
    df['influencer_type'] = df['source']


Running on device: cpu


In [8]:
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()

video_id_col = 'video_id' if 'video_id' in df.columns else 'id'


def get_video_id(row) -> str:
    value = row.get(video_id_col, None)
    if pd.isna(value):
        return ''
    return str(value)


def get_duration_seconds(row):
    for col in ('video_duration_seconds', 'duration_seconds', 'duration', 'video_duration'):
        if col in row.index:
            value = row.get(col, np.nan)
            if pd.notna(value):
                return value
    return np.nan


def has_frames(video_id: str) -> bool:
    return (FRAME_ROOT / video_id).is_dir()


video_ids = df[video_id_col].astype(str)
df['has_frames'] = [has_frames(video_id) for video_id in video_ids]

missing_ids = video_ids[~df['has_frames']].tolist()
print(f'Total videos in CSV: {len(df)}')
print(f'Videos with frame folder: {df["has_frames"].sum()}')
print(f'Videos missing frame folder: {len(missing_ids)}')
if missing_ids:
    print('First missing video_ids:', missing_ids[:20])

df = df[df['has_frames']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available frames')


Total videos in CSV: 500
Videos with frame folder: 500
Videos missing frame folder: 0
Restricted to 500 rows with available frames


In [9]:
def download_if_missing(url: str, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if not out_path.exists():
        print(f'Downloading {out_path.name}...')
        urlretrieve(url, out_path)


def load_places365_categories(categories_file: Path):
    with open(categories_file, 'r', encoding='utf-8') as f:
        return [line.strip().split(' ')[0][3:] for line in f]


def load_places365_resnet18(weights_path: Path):
    scene_model = models.resnet18(num_classes=365)
    checkpoint = torch.load(weights_path, map_location='cpu')
    state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint
    clean_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    scene_model.load_state_dict(clean_state_dict)
    scene_model.to(device)
    scene_model.eval()
    return scene_model


def sample_frame_paths(video_id: str, duration_seconds=None, default_max_frames: int = DEFAULT_MAX_FRAMES_PER_VIDEO):
    folder = FRAME_ROOT / video_id
    if not folder.is_dir():
        return []

    frame_files = sorted(folder.glob('*.jpeg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.jpg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.png'))
    if not frame_files:
        return []

    try:
        duration_value = float(duration_seconds)
    except (TypeError, ValueError):
        duration_value = np.nan

    if pd.notna(duration_value) and duration_value > 0:
        target_frames = int(np.ceil(duration_value))
    else:
        target_frames = default_max_frames

    target_frames = max(1, min(target_frames, len(frame_files)))
    step = max(1, len(frame_files) // target_frames)
    return frame_files[::step][:target_frames]


def classify_scene(image_path: Path):
    try:
        image = Image.open(image_path).convert('RGB')
        x = transform(image).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            conf, idx = probs.max(dim=1)
        return classes[idx.item()], float(conf.item())
    except Exception as e:
        print(f'Error reading {image_path}: {e}')
        return None, np.nan


download_if_missing(MODEL_URL, MODEL_WEIGHTS)
download_if_missing(CATEGORIES_URL, CATEGORIES_TXT)

classes = load_places365_categories(CATEGORIES_TXT)
model = load_places365_resnet18(MODEL_WEIGHTS)

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f'Loaded Places365 model with {len(classes)} classes')


Loaded Places365 model with 365 classes


In [10]:
video_scene_top1 = []
video_scene_top1_conf = []
video_scene_unique_labels = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Processing videos'):
    vid = get_video_id(row)
    duration_seconds = get_duration_seconds(row)
    frame_paths = sample_frame_paths(vid, duration_seconds=duration_seconds)

    labels = []
    confidences = []

    for fp in frame_paths:
        label, conf = classify_scene(fp)
        if label is not None and not np.isnan(conf):
            labels.append(label)
            confidences.append(conf)

    if labels:
        counts = Counter(labels)
        top_label, _ = counts.most_common(1)[0]
        top_confs = [c for l, c in zip(labels, confidences) if l == top_label]
        top_conf = float(np.mean(top_confs)) if top_confs else float(np.mean(confidences))

        video_scene_top1.append(top_label)
        video_scene_top1_conf.append(top_conf)
        video_scene_unique_labels.append(len(counts))
    else:
        video_scene_top1.append('Unbestimmt')
        video_scene_top1_conf.append(np.nan)
        video_scene_unique_labels.append(np.nan)


Processing videos: 100%|██████████| 500/500 [07:22<00:00,  1.13it/s]


In [11]:
df['scene_top1_label'] = video_scene_top1
df['scene_top1_confidence'] = video_scene_top1_conf
df['scene_unique_labels'] = video_scene_unique_labels

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f'Saved: {OUTPUT_CSV}')
df[['scene_top1_label', 'scene_top1_confidence', 'scene_unique_labels']].head(20)


Saved: ..\..\data\04_analysis_results\visual_features\06_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_scene_classification.csv


,scene_top1_label,scene_top1_confidence,scene_unique_labels
0,beauty_salon,0.463629,1
1,beauty_salon,0.347606,4
2,jewelry_shop,0.201395,2
3,beer_garden,0.465769,3
4,beauty_salon,0.650550,2
5,living_room,0.209752,2
6,desert/sand,0.344681,3
7,bathroom,0.530247,1
8,television_studio,0.313111,3
9,beauty_salon,0.569897,1
